In [1]:
import numpy as np
import pickle
import pandas as pd
import time

In [2]:
def generate_user_contexts(num_iter, rounds_per_iter, dim_embedding, noise_strength, publisher_embeddings):
    """
    Genera un dizionario con publisher come chiave e un array con dimensioni num_iter x rounds_per_iter x dim_embedding
    """
    n, m, k = num_iter, rounds_per_iter, dim_embedding
    user_contexts = {}
    for publisher_name, pub_emb in publisher_embeddings.items():
        pub_array = np.array(pub_emb, dtype=np.float32)
        new_array = np.tile(pub_array, (n, m, 1))
        noise_array = np.random.normal(0, noise_strength, size=(n, m, k)).astype(np.float32)
        new_array += noise_array
        user_contexts[publisher_name] = new_array
    return user_contexts

def compute_sigmoids(user_contexts, adv_embeddings):
    """
    Restituisco un dizionario con publisher come chiave e un dizionario con annunci come
    chiave e una matrice numpy con dimensioni num_iter x rounds_per_iter come valore
    """
    # Compute scalar products
    scalar_products = {}
    for publisher_name, user_context in user_contexts.items():
        scalar_products[publisher_name] = {}
        for adv_name, adv_emb in adv_embeddings.items():
            adv_array = np.array(adv_emb, dtype=np.float32)
            scalar_products[publisher_name][adv_name] = np.einsum('ijk,k->ij', user_context, adv_array)
    # Compute mean and std over all scalar products for all publishers and ads
    all_tensors = np.concatenate([score for adv_scores in scalar_products.values() for score in adv_scores.values()])
    mean_scores = np.mean(all_tensors)
    std_scores = np.std(all_tensors)
    # Compute sigmoids
    sigmoids = {}
    for publisher_name, adv_scores in scalar_products.items():
        sigmoids[publisher_name] = {}
        for adv_name, score in adv_scores.items():
            sigmoids[publisher_name][adv_name] = 1 / (1 + np.exp(-(score - mean_scores) / (0.5 * std_scores)))
    return sigmoids

def initialize_deal(num_iter, rounds_per_iter, dim_embedding, noise_strength, publisher_embeddings, adv_embeddings):
    user_contexts = generate_user_contexts(num_iter, rounds_per_iter, dim_embedding, noise_strength, publisher_embeddings)
    adv_sigmoids = compute_sigmoids(user_contexts, adv_embeddings)
    return user_contexts, adv_sigmoids

In [3]:
def get_partecipant_mask(A, num_participants_per_round, rng):
    # Dimensioni
    n, m, p, r = A.shape
    # Numero totale di elementi nelle dimensioni (n, m, p)
    total_positions = n * m * p

    # Genera numeri casuali per ordinamento
    random_numbers = rng.random((total_positions, r)).astype(np.float32)

    # Ordina i numeri casuali lungo l'ultima dimensione
    sorted_indices = np.argsort(random_numbers, axis=1)

    # Prendi i primi num_participants_per_round indici per ciascun gruppo
    selected_indices = sorted_indices[:, :num_participants_per_round]

    # Prepara maschera finale
    mask = np.zeros((n, m, p, r), dtype=np.int8)

    # Crea array di indici per le prime tre dimensioni
    i_indices, j_indices, k_indices = np.meshgrid(
        np.arange(n), np.arange(m), np.arange(p), indexing='ij'
    )

    # Flatten per allineare agli indici selezionati
    i_indices = i_indices.ravel()
    j_indices = j_indices.ravel()
    k_indices = k_indices.ravel()

    # Assegna i valori 1 alla maschera usando gli indici selezionati
    for idx in range(total_positions):
        mask[i_indices[idx], j_indices[idx], k_indices[idx], selected_indices[idx]] = 1

    return mask

def simulate_auctions(
    publisher_embeddings: dict,
    adv_embeddings: dict,
    pub_list: list,
    num_iter: int,
    rounds_per_iter: int,
    num_participants_per_round: int,
    noise_std: float = 0.01,
    rng: np.random.Generator = None
) -> pd.DataFrame:
    print(f"Starting simulation with {num_iter} iterations and {rounds_per_iter} rounds per iteration. Number of publishers: {len(pub_list)}. Number of advertisers: {len(adv_embeddings)}")
    if rng is None:
        rng = np.random.default_rng()
        
    # Filter publishers
    t0 = time.time()
    
    selected_publishers_embeddings = {pub: publisher_embeddings[pub] for pub in pub_list}
    
    print(f"Filtering publishers took: {time.time() - t0:.4f} seconds")
    
    # Convert to array the dicts
    t1_a = time.time()
    
    publisher_embeddings_array = np.array(list(selected_publishers_embeddings.values()), dtype=np.float32)
    
    print(f"Converting arrays took: {time.time() - t1_a:.4f} seconds")
    t1_b = time.time()
    
    publisher_embeddings_array_tiled = np.tile(publisher_embeddings_array, 
                                             (num_iter, rounds_per_iter, 1, 1))
    
    print(f"Tiling arrays took: {time.time() - t1_b:.4f} seconds")
    t1_c = time.time()

    # Create and add noise

    publisher_embeddings_array_tiled = rng.normal(
        loc=publisher_embeddings_array_tiled,  # usa l'array esistente come media
        scale=noise_std,                        
        size=None,                             # usa la dimensione dell'array esistente
    ).astype(np.float32)

    print(f"Generating an adding noise took: {time.time() - t1_c:.4f} seconds")
    # Convert advertiser embeddings and compute scalar products
    t2 = time.time()
    
    adv_embeddings_array = np.array(list(adv_embeddings.values()), dtype=np.float32)
    
    # Reshape for better cache utilization
    pub_shape = publisher_embeddings_array_tiled.shape
    publisher_embeddings_array_tiled = publisher_embeddings_array_tiled.reshape(-1, pub_shape[-1])
    scalar_products = np.dot(publisher_embeddings_array_tiled, adv_embeddings_array.T)
    scalar_products = scalar_products.reshape(pub_shape[0], pub_shape[1], pub_shape[2], -1)

    print(f"Computing scalar products took: {time.time() - t2:.4f} seconds")
    
    # Compute sigmoids
    t3 = time.time()
    
    mean_scores = np.mean(scalar_products)
    std_scores = np.std(scalar_products)
    sigmoids = 1 / (1 + np.exp(-(scalar_products - mean_scores) / (0.5 * std_scores)))
    
    print(f"Computing sigmoids took: {time.time() - t3:.4f} seconds")
    
    # Get participant mask
    t4_a = time.time()

    partecipant_mask = get_partecipant_mask(sigmoids, num_participants_per_round, rng)
    
    print(f"Getting participant mask took: {time.time() - t4_a:.4f} seconds")
    
    t4_b = time.time()
    
    partecipant_sigmoids = sigmoids * partecipant_mask
    
    print(f"Multiplying sigmoids took: {time.time() - t4_b:.4f} seconds")
    
    # Determine winners and calculate CTR/impressions
    t5 = time.time()
    
    winners = np.argmax(partecipant_sigmoids, axis=3)
    our_sigmoids = sigmoids[:,:,:,0]
    our_ctr = np.where((winners==0), our_sigmoids, 0)
    our_clicks = rng.binomial(1, our_ctr).sum(axis=1)
    our_impressions = (winners==0).astype(np.int32)
    
    print(f"Determining winners and calculating CTR took: {time.time() - t5:.4f} seconds")
    
    # Aggregate results over the rounds of each iteration
    clicks = our_ctr.sum(axis=1)
    impressions = our_impressions.sum(axis=1)
    
    results = pd.DataFrame()
    for i in range(num_iter):
        curr_results = pd.DataFrame({
            'publisher': pub_list,
            'Iteration': i,
            'true_clicks': clicks[i],
            'clicks': our_clicks[i],
            'impressions': impressions[i]
        })
        results = pd.concat([results, curr_results])
    
    group_pub_res = pd.DataFrame({
        'publisher': pub_list,
        'clicks': clicks.mean(axis=0),
        'impressions': impressions.mean(axis=0)
    })
    
    return results, group_pub_res

In [4]:
def simulate_auctions_from_oldgen(sigmoids, pub_list, num_iter, num_participants_per_round, rng):
    partecipant_mask = get_partecipant_mask(sigmoids, num_participants_per_round, rng)
    
    partecipant_sigmoids = sigmoids * partecipant_mask
    
    # Determine winners and calculate CTR/impressions
    
    winners = np.argmax(partecipant_sigmoids, axis=3)
    our_sigmoids = sigmoids[:,:,:,0]
    our_ctr = np.where((winners==0), our_sigmoids, 0)
    our_clicks = rng.binomial(1, our_ctr).sum(axis=1)
    our_impressions = (winners==0).astype(np.int32)
    
    # Aggregate results over the rounds of each iteration
    clicks = our_ctr.sum(axis=1)
    impressions = our_impressions.sum(axis=1)
    
    results = pd.DataFrame()
    for i in range(num_iter):
        curr_results = pd.DataFrame({
            'publisher': pub_list,
            'Iteration': i,
            'true_clicks': clicks[i],
            'clicks': our_clicks[i],
            'impressions': impressions[i]
        })
        results = pd.concat([results, curr_results])
    
    group_pub_res = pd.DataFrame({
        'publisher': pub_list,
        'clicks': clicks.mean(axis=0),
        'impressions': impressions.mean(axis=0)
    })
    
    return results, group_pub_res

In [5]:
# Set up Random Number Generator
seed = 0
rng = np.random.default_rng(seed)
np.random.seed(seed)

In [6]:
num_iter = 100
rounds_per_iter = 200
dim_embedding = 70
noise_strength = 0.01
num_participants_per_round = 4

In [8]:
## Carico adv e publisher embedding
adv_embedding_path = '../src/publisher_embedding/data/embeddings_to_pick/ad_embeddings_red_70.pkl'
adv_embeddings = pickle.load(open(adv_embedding_path, 'rb'))

In [9]:
agents = [
  {
    "name": "Nostro",
    "adv_name": "Racer 1000",
  },
  {
    "name": "Reflex Pro",
    "adv_name": "Reflex Pro",
  },
  {
    "name": "Speedster GT",
    "adv_name": "Speedster GT",
  },
  {
    "name": "Pasta Bio",
    "adv_name": "Pasta Bio",
  },
  {
    "name": "Sneakers High-top",
    "adv_name": "Sneakers High-top"
  },
  {
    "name": "Conto Risparmio Plus",
    "adv_name": "Conto Risparmio Plus",
  },
  {
    "name": "Smartphone Pro X",
    "adv_name": "Smartphone Pro X",
  }
]

In [10]:
selected_adv_embeddings = {agent['adv_name']: adv_embeddings[agent['adv_name']] for agent in agents}

In [11]:
publisher_embeddings_path = '../src/publisher_embedding/data/embeddings_to_pick/sites_embeddings_red_70.pkl'
publisher_embeddings = pickle.load(open(publisher_embeddings_path, 'rb'))

### Filtro publisher da simulare

In [8]:
# Read 1 run file of an experiment
exp_path = '../results/FP_Truthful_Oracle_sigmoids_linucb_rescaledtextemb_ctr_0_9_10_12_24_nuovipub/'
run_path = exp_path + 'agent_stats_run_0_ctr_0.9_alpha_1.csv'
run_df = pd.read_csv(run_path)
# Take only the publishers of last iteration
last_iter = 0
pub_list = run_df[run_df['Iteration']==last_iter]['publisher'].tolist()
print(f"Number of publishers: {len(pub_list)}")

Number of publishers: 287


## Generazione dati con dict (versione iniziale)

In [16]:
filter_embeddings = {pub: publisher_embeddings[pub] for pub in pub_list}

In [55]:
user_contexts, adv_sigmoids = initialize_deal(num_iter, rounds_per_iter, dim_embedding, noise_strength, filter_embeddings, selected_adv_embeddings)

### Per confronto filtro solo i sigmoidi del mio advertiser

In [56]:
sigmoids_oldgen = np.array([ [adv_sigmoids[pub][agent['adv_name']] for agent in agents] for pub in pub_list])

In [57]:
sigmoids_oldgen.shape

(287, 7, 100, 200)

In [58]:
# Remove the singleton dimension and transpose the axes
# I have (num_pub, num_agents, num_iter, rounds_per_iter) and I want (num_iter, rounds_per_iter, num_pub, num_agents)
sigmoids_oldgen = sigmoids_oldgen.squeeze().transpose(2, 3, 0, 1)

In [59]:
sigmoids_oldgen.shape

(100, 200, 287, 7)

In [60]:
# Verify that the new function works
assert sigmoids_oldgen[0, 0, 0, 0] == adv_sigmoids[pub_list[0]][agents[0]['adv_name']][0, 0]

In [61]:
results_old_gen, group_pub_res_old_gen = simulate_auctions_from_oldgen(sigmoids_oldgen, pub_list, num_iter, num_participants_per_round, rng)

## Generazione dati con numpy array (versione attuale)

In [62]:
sim_auctions, group_pub_res = simulate_auctions(
    publisher_embeddings=publisher_embeddings,
    adv_embeddings=selected_adv_embeddings,
    pub_list=pub_list,
    num_iter=num_iter,
    rounds_per_iter=rounds_per_iter,
    num_participants_per_round=num_participants_per_round,
    noise_std=0.01,
    rng=rng
)

Starting simulation with 100 iterations and 200 rounds per iteration. Number of publishers: 287. Number of advertisers: 7
Filtering publishers took: 0.0001 seconds
Converting arrays took: 0.0008 seconds
Tiling arrays took: 0.1725 seconds
Generating an adding noise took: 8.4486 seconds
Computing scalar products took: 0.1249 seconds
Computing sigmoids took: 0.1997 seconds
Getting participant mask took: 9.2462 seconds
Multiplying sigmoids took: 0.0251 seconds
Determining winners and calculating CTR took: 0.2888 seconds


In [63]:
group_pub_res_old_gen.head()

,publisher,clicks,impressions
0,eltiempo.es,35.008980,51.57
1,altovicentinonline.it,49.159740,57.67
2,elchapuzasinformatico.com,53.450882,57.02
3,dating.lovepedia.net,2.759545,26.84
4,espabox.com,32.580647,60.23


In [64]:
group_pub_res.head()

,publisher,clicks,impressions
0,eltiempo.es,36.202820,53.25
1,altovicentinonline.it,49.753330,58.34
2,elchapuzasinformatico.com,54.017040,57.60
3,dating.lovepedia.net,2.703148,26.40
4,espabox.com,32.500813,60.12


### Check the mean clicks and impressions differences between the two methods

In [65]:
compare_gen = pd.merge(group_pub_res_old_gen, group_pub_res, on='publisher', suffixes=('_old_gen', '_new_gen'))

compare_gen['clicks_diff'] = np.abs(compare_gen['clicks_new_gen'] - compare_gen['clicks_old_gen'])
compare_gen['impressions_diff'] = np.abs(compare_gen['impressions_new_gen'] - compare_gen['impressions_old_gen'])

mean_clicks_diff = compare_gen['clicks_diff'].mean()
mean_impressions_diff = compare_gen['impressions_diff'].mean()

print(f"Mean difference in clicks: {mean_clicks_diff}")
print(f"Mean difference in impressions: {mean_impressions_diff}")

Mean difference in clicks: 0.5439357161521912
Mean difference in impressions: 0.6854703832752614


### Apply Knapsack solver to compare optimal publishers

In [66]:
from ortools.linear_solver import pywraplp

def get_data(
        df: pd.DataFrame,
) -> tuple[int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n = df.shape[0]
    clicks = df['clicks'].values
    impressions = df['impressions'].values
    return n, clicks, impressions

def solver(
        df: pd.DataFrame,
        n: int,
        clicks: np.ndarray,
        impressions: np.ndarray,
        soglia_ctr: float = None,
) -> pd.DataFrame:
    solver = pywraplp.Solver.CreateSolver('SCIP')
    # Boolean variables
    x = [solver.BoolVar(f'x{i}') for i in range(n)]
    x_np = np.array(x)
    # Objective function
    solver.Maximize(np.dot(clicks, x_np))
    if soglia_ctr is not None:
        # CTR constraint
        solver.Add(np.dot(clicks, x_np) >= soglia_ctr * np.dot(impressions, x_np))
    # Solve the knapsack problem
    status = solver.Solve()
    results = pd.DataFrame(columns=df.columns)
    # Cycle over the boolean variables to get the selected rows
    if status == pywraplp.Solver.OPTIMAL:
        for i in range(n):
            if x[i].solution_value() == 1:
                if results.empty:
                    results = df.iloc[[i]]
                else:
                    results = pd.concat([results, df.iloc[[i]]])
        print("Knapsack Solver: Optimal solution!")
        print(f"Total clicks = {results['clicks'].sum()}")
        print(f"Total impressions = {results['impressions'].sum()}")
        print(f"Number of selected publishers = {results.shape[0]}")
        if soglia_ctr is not None:
            if results['impressions'].sum() != 0:
                print(f"CTR = {results['clicks'].sum() / results['impressions'].sum()}")
            else:
                print("CTR = undefined (division by zero)")
    else:
        print("Knapsack Solver: No feasible solution found.")
        # Return empty dataframe
        return pd.DataFrame()
    return results

def knapsack(
        df: pd.DataFrame,
        soglia_ctr: float = None,
) -> pd.DataFrame:
    n, clicks, impressions = get_data(df)
    return solver(df, n, clicks, impressions, soglia_ctr)

In [67]:
opt_results_oldgen = knapsack(group_pub_res_old_gen, soglia_ctr=0.9)

Knapsack Solver: Optimal solution!
Total clicks = 11178.24609375
Total impressions = 12420.259999999998
Number of selected publishers = 145
CTR = 0.9000009737115006


In [68]:
opt_results_newgen = knapsack(group_pub_res, soglia_ctr=0.9)

Knapsack Solver: Optimal solution!
Total clicks = 11172.4765625
Total impressions = 12413.45
Number of selected publishers = 145
CTR = 0.9000299322509052


In [70]:
opt_publisher_list = opt_results_oldgen['publisher'].tolist()

### Verifica con esperimento

In [27]:
exp_res = pd.read_csv('../results/FP_Truthful_Oracle_sigmoids_cucb_ctr_0_9_30_01_25_prova/agent_stats_run_0_ctr_0.9_alpha_1.csv')

In [15]:
# filtro solo i publisher ottimi
exp_res = exp_res[exp_res['publisher'].isin(opt_publisher_list)]
exp_res = exp_res.groupby('publisher').agg({'clicks': 'mean', 'impressions': 'mean'}).reset_index()

NameError: name 'opt_publisher_list' is not defined

In [16]:
exp_res.head()

,publisher,impressions,lost_auctions,win_rate,clicks,true_clicks,ctr,true_ctr,spent,mean_bid,cpc,cpm,Agent,Iteration,Run,est_clicks,est_impressions,conf_bound
0,ara.cat,106,1,0.990654,100,98.531114,94.339623,0.929539,98.531110,0.929539,0.985311,929.538799,Nostro 1,0,0,NaN,NaN,NaN
1,prosport.ro,73,38,0.657658,63,61.386196,86.301370,0.840907,61.386196,0.840907,0.974384,840.906796,Nostro 1,0,0,NaN,NaN,NaN
2,juegos.elpais.com,101,26,0.795276,83,77.108736,82.178218,0.763453,77.108740,0.763453,0.929021,763.452889,Nostro 1,0,0,NaN,NaN,NaN
3,meteofrance.com,19,88,0.177570,17,14.279789,89.473684,0.751568,14.279789,0.751568,0.839988,751.567841,Nostro 1,0,0,NaN,NaN,NaN
4,labaroviola.com,43,64,0.401869,29,33.947857,67.441860,0.789485,33.947860,0.789485,1.170616,789.485133,Nostro 1,0,0,NaN,NaN,NaN


In [28]:
last_iter_exp = exp_res[exp_res['Iteration']==99]
last_iter_exp['clicks'].sum() / last_iter_exp['impressions'].sum()

0.9061910478477105

In [29]:
last_iter_exp['clicks'].sum()

10568

In [20]:
group_pub_res[group_pub_res['publisher']=='1001juegos.com']

NameError: name 'group_pub_res' is not defined

In [11]:
run_df[run_df['Iteration']==499]['clicks'].sum()

13251.0